# **[crawl4ai](https://docs.crawl4ai.com/core/installation/)**

## 🚀🤖 Crawl4AI: **Open-Source LLM-Friendly Web Crawler & Scraper**

In [1]:
%%capture
! pip install crawl4ai -q

In [2]:
%%capture
! crawl4ai-setup -q

In [3]:
%%capture
! crawl4ai-doctor -q

## **[0. quickstart](https://docs.crawl4ai.com/core/quickstart/)**

In [25]:
import asyncio
from crawl4ai import AsyncWebCrawler

async def main():
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun("https://example.com")
        print(result.markdown)  # Print first 300 chars

if __name__ == "__main__":
    asyncio.run(main())


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://example.com                                                                                  |
✓ | ⏱: 0.58s 

[SCRAPE].. ◆ https://example.com                                                                                  |
✓ | ⏱: 0.00s 

[COMPLETE] ● https://example.com                                                                                  |
✓ | ⏱: 0.59s 

# Example Domain
This domain is for use in documentation examples without needing permission. Avoid use in operations.
[Learn more](https://iana.org/domains/example)



### 1. 정적(Static) 웹사이트 내용 Scrapping

In [1]:
import nest_asyncio
import asyncio
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig

nest_asyncio.apply()  # 기존 루프에 중첩 실행 허용

# 글로벌 변수 선언
result = None

async def main():
    global result  # 전역 변수로 지정
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            # url="https://www.example.com",
            url ='https://quotes.toscrape.com/'
        )
        print(result.markdown[:300])

        return result.markdown

raw_text = await main()

# 이후 result를 자유롭게 사용 가능
# 예: print(result.html) 또는 result.title 등


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://quotes.toscrape.com/                                                                         |
✓ | ⏱: 1.27s 

[SCRAPE].. ◆ https://quotes.toscrape.com/                                                                         |
✓ | ⏱: 0.06s 

[COMPLETE] ● https://quotes.toscrape.com/                                                                         |
✓ | ⏱: 1.37s 

#  [Quotes to Scrape](https://quotes.toscrape.com/)
[Login](https://quotes.toscrape.com/login)
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” by Albert Einstein [(about)](https://quotes.toscrape.com/author/Albert-Einstein)
Tags: [c


In [10]:
raw_text

"#  [Quotes to Scrape](https://quotes.toscrape.com/)\n[Login](https://quotes.toscrape.com/login)\n“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” by Albert Einstein [(about)](https://quotes.toscrape.com/author/Albert-Einstein)\nTags: [change](https://quotes.toscrape.com/tag/change/page/1/) [deep-thoughts](https://quotes.toscrape.com/tag/deep-thoughts/page/1/) [thinking](https://quotes.toscrape.com/tag/thinking/page/1/) [world](https://quotes.toscrape.com/tag/world/page/1/)\n“It is our choices, Harry, that show what we truly are, far more than our abilities.” by J.K. Rowling [(about)](https://quotes.toscrape.com/author/J-K-Rowling)\nTags: [abilities](https://quotes.toscrape.com/tag/abilities/page/1/) [choices](https://quotes.toscrape.com/tag/choices/page/1/)\n“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” by Albert Einstein [(about)](ht

### final 내용을 복사하여 Gemini에게 마크다운 형식을 정리하는 코드를 달라고 하면

In [11]:
# 2. 데이터 추출을 위한 정규표현식
# 명언과 저자 추출 (링크 구조 [(about)](...) 포함 대응)
quote_pattern = r'“(.+?)” by ([\w\s\.-]+)'
# 태그 추출 ([태그명](링크) 구조에서 태그명만 추출)
tag_pattern = r'Tags: (.+)'

# 3. 데이터 파싱
quotes_found = re.findall(quote_pattern, raw_text)
tags_found = re.findall(tag_pattern, raw_text)

data = []
for i in range(len(quotes_found)):
    quote_text = quotes_found[i][0]
    author = quotes_found[i][1].strip()

    # 태그에서 [태그명]만 추출하여 쉼표로 연결
    if i < len(tags_found):
        raw_tags = re.findall(r'\[(.+?)\]', tags_found[i])
        tag_str = ", ".join(raw_tags)
    else:
        tag_str = "None"

    data.append({
        "Quote": quote_text,
        "Author": author,
        "Tags": tag_str
    })

# 4. 결과 출력
df = pd.DataFrame(data)
df

,Quote,Author,Tags
0,The world as we have created it is a process o...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"abilities, choices"
2,There are only two ways to live your life. One...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"The person, be it gentleman or lady, who has n...",Jane Austen,"aliteracy, books, classic, humor"
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"be-yourself, inspirational"
5,Try not to become a man of success. Rather bec...,Albert Einstein,"adulthood, success, value"
6,It is better to be hated for what you are than...,André Gide,"life, love"
7,"I have not failed. I've just found 10,000 ways...",Thomas A. Edison,"edison, failure, inspirational, paraphrased"
8,A woman is like a tea bag; you never know how ...,Eleanor Roosevelt,misattributed-eleanor-roosevelt
9,"A day without sunshine is like, you know, night.",Steve Martin,"humor, obvious, simile"


In [12]:
import re

# final 변수에 HTML 전체 텍스트가 저장되어 있다고 가정
pattern = r'“(.*?)” by ([\w\.\-\' ]+)'

matches = re.findall(pattern, raw_text)

# 결과 확인
for quote, author in matches:
    print(f'"{quote}" - {author}')


"The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking." - Albert Einstein 
"It is our choices, Harry, that show what we truly are, far more than our abilities." - J.K. Rowling 
"There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle." - Albert Einstein 
"The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid." - Jane Austen 
"Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring." - Marilyn Monroe 
"Try not to become a man of success. Rather become a man of value." - Albert Einstein 
"It is better to be hated for what you are than to be loved for what you are not." - André Gide 
"I have not failed. I've just found 10,000 ways that won't work." - Thomas A. Edison 
"A woman is like a tea bag; you never know how strong it is until it's in hot water." - Elea

In [2]:
result.html

'<!DOCTYPE html><html lang="en"><head>\n\t<meta charset="UTF-8">\n\t<title>Quotes to Scrape</title>\n    <link rel="stylesheet" href="/static/bootstrap.min.css">\n    <link rel="stylesheet" href="/static/main.css">\n    \n    \n</head>\n<body>\n    <div class="container">\n        <div class="row header-box">\n            <div class="col-md-8">\n                <h1>\n                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>\n                </h1>\n            </div>\n            <div class="col-md-4">\n                <p>\n                \n                    <a href="/login">Login</a>\n                \n                </p>\n            </div>\n        </div>\n    \n\n<div class="row">\n    <div class="col-md-8">\n\n    <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">\n        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>\n

In [3]:
from bs4 import BeautifulSoup

# body 태그 내 텍스트만 추출
soup = BeautifulSoup(result.html, "html.parser")
body_text = soup.body.get_text(separator="\n", strip=True)

print(body_text)


Quotes to Scrape
Login
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
by
Albert Einstein
(about)
Tags:
change
deep-thoughts
thinking
world
“It is our choices, Harry, that show what we truly are, far more than our abilities.”
by
J.K. Rowling
(about)
Tags:
abilities
choices
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
by
Albert Einstein
(about)
Tags:
inspirational
life
live
miracle
miracles
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
by
Jane Austen
(about)
Tags:
aliteracy
books
classic
humor
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
by
Marilyn Monroe
(about)
Tags:
be-yourself
inspirational
“Try not to become a man of success. Rather become a man of value.”
by
Albert Einstein
(about)
Tags:
adulthood
success

In [4]:
print(soup)

<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8"/>
<title>Quotes to Scrape</title>
<link href="/static/bootstrap.min.css" rel="stylesheet"/>
<link href="/static/main.css" rel="stylesheet"/>
</head>
<body>
<div class="container">
<div class="row header-box">
<div class="col-md-8">
<h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
</div>
<div class="col-md-4">
<p>
<a href="/login">Login</a>
</p>
</div>
</div>
<div class="row">
<div class="col-md-8">
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="ta

### 2. 동적(Dynamic) 웹사이트 내용 Scrapping
- Rendered HTML의 내용을 직접 추출

In [7]:
! pip install html2text -q

In [8]:
import asyncio
from playwright.async_api import async_playwright
import html2text

url = "https://comic.naver.com/webtoon"

async def test_browser():
    async with async_playwright() as p:
        # 1. 브라우저 실행
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # 2. 페이지 이동 및 대기
        await page.goto(url)
        # 네이버 웹툰은 동적 로딩이 많으므로 네트워크가 조용해질 때까지 기다리는 것이 좋습니다.
        await page.wait_for_load_state("networkidle")

        # 3. HTML 소스 가져오기
        html_content = await page.content()

        # 4. 마크다운으로 변환하기
        h = html2text.HTML2Text()
        h.ignore_links = False  # 링크 포함 여부 설정
        markdown_content = h.handle(html_content)

        # 5. 결과 확인 (앞부분 일부만 출력)
        print(f"--- Title: {await page.title()} ---")

        print("\n[HTML 소스 상위 500자]")
        print(html_content[:500])

        print("\n[마크다운 변환 결과 상위 500자]")
        print(markdown_content[:500])

        # 6. 파일로 저장하기
        with open("webtoon.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        with open("webtoon.md", "w", encoding="utf-8") as f:
            f.write(markdown_content)

        await browser.close()

if __name__ == "__main__":
    asyncio.run(test_browser())

--- Title: 요일전체 : 네이버 웹툰 ---

[HTML 소스 상위 500자]
<!DOCTYPE html><html lang="ko"><head>
	<title>요일전체 : 네이버 웹툰</title>
	<link rel="icon" href="https://shared-comic.pstatic.net/favicon/favicon_96x96.ico" type="image/x-icon">


	<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/static/wcc/kw-owner/prod-1.0/index.css?v=202604071940">

	<link rel="canonical" href="https://m.comic.naver.com/webtoon/weekday">
	<meta name="google-site-verification" content="6Wv2YrpTSpapViVikuUS-ebAqZDiKxcrDV_clkRzH9A">
	<meta charset="utf-8">
	<meta 

[마크다운 변환 결과 상위 500자]
메인메뉴로 바로가기본문으로 바로가기

#
[NAVER](https://naver.com)[웹툰](/index)|[웹소설](https://novel.naver.com)|[시리즈](https://series.naver.com)

검색

* * *

  * [홈](/index)
  * [웹툰](/webtoon)
  *  _NEW_ 웹툰의 또다른 플레이[컷츠](/cuts/v)
  * [베스트도전](/bestChallenge)
  * [도전만화](/challenge)
  * [마이페이지](/mypage/favorite)

 _CREATOR'S_ _NEW_

 __

**숏폼 서비스 컷츠 출시!**

웹툰 올리기 뿐만 아니라 컷츠 업로드와 관리도 가능합니다.  
달라진 창작 환경을 경험해 보세요.

툴팁 닫기

* * *

  * [요일전체](/webtoon)
  * 

In [9]:
import asyncio
from playwright.async_api import async_playwright
import html2text

url = "https://comic.naver.com/webtoon"

async def test_browser():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        # 실제 브라우저와 유사한 환경을 위해 viewport 크기 설정
        page = await browser.new_page(viewport={'width': 1280, 'height': 1024})

        await page.goto(url)

        # 핵심: '이달의 신규 웹툰' 섹션이나 특정 웹툰 아이템이 로드될 때까지 대기
        # .ContentList__content_list--_S99K 는 네이버 웹툰의 리스트 클래스 예시입니다.
        try:
            # 1. 특정 셀렉터가 나타날 때까지 최대 10초 대기
            await page.wait_for_selector("[class*='ContentList__content_list']", timeout=10000)

            # 2. 동적 로딩을 위해 페이지를 아래로 살짝 스크롤 (필요한 경우)
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight/2)")
            await asyncio.sleep(2) # 렌더링 시간 확보

        except Exception as e:
            print(f"대기 중 오류 발생 또는 타임아웃: {e}")

        # HTML 가져오기
        html_content = await page.content()

        # 마크다운 변환
        h = html2text.HTML2Text()
        h.body_width = 0 # 줄바꿈 방지
        markdown_content = h.handle(html_content)

        # 파일 저장
        with open("naver_webtoon.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        with open("naver_webtoon.md", "w", encoding="utf-8") as f:
            f.write(markdown_content)

        print("스크래핑 완료: naver_webtoon.html, naver_webtoon.md 저장됨")
        await browser.close()

asyncio.run(test_browser())

대기 중 오류 발생 또는 타임아웃: Page.wait_for_selector: Timeout 10000ms exceeded.
Call log:
  - waiting for locator("[class*='ContentList__content_list']") to be visible

스크래핑 완료: naver_webtoon.html, naver_webtoon.md 저장됨


In [15]:
import requests
from bs4 import BeautifulSoup

# 파일 경로
file_path = "/content/webtoon.html"

# 파일을 읽고 BeautifulSoup으로 파싱
try:
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    soup = BeautifulSoup(html_content, "lxml")  # 또는 "html.parser"
    print("HTML content loaded and parsed successfully.")
    # Optionally, print a part of the soup to confirm
    print(soup.prettify()[:3000]) # Print first 1000 characters for a cleaner output
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

HTML content loaded and parsed successfully.
<!DOCTYPE html>
<html lang="ko">
 <head>
  <title>
   요일전체 : 네이버 웹툰
  </title>
  <link href="https://shared-comic.pstatic.net/favicon/favicon_96x96.ico" rel="icon" type="image/x-icon"/>
  <link href="https://ssl.pstatic.net/static/wcc/kw-owner/prod-1.0/index.css?v=202604071940" rel="stylesheet" type="text/css"/>
  <link href="https://m.comic.naver.com/webtoon/weekday" rel="canonical"/>
  <meta content="6Wv2YrpTSpapViVikuUS-ebAqZDiKxcrDV_clkRzH9A" name="google-site-verification"/>
  <meta charset="utf-8"/>
  <meta content="ie=edge" http-equiv="x-ua-compatible"/>
  <meta content="article" property="og:type"/>
  <meta content="네이버 웹툰" property="og:article:author"/>
  <meta content="https://comic.naver.com" property="og:article:author:url"/>
  <meta content="네이버 웹툰" property="og:title"/>
  <meta content="https://ssl.pstatic.net/static/comic/images/og_tag_v3.png" property="og:image"/>
  <meta content="매일매일 새로운 재미, 네이버 웹툰." property="og:descriptio

### 3. [Browser, Crawler & LLM Configuration](https://docs.crawl4ai.com/core/browser-crawler-config/)

- 웹 크롤링 라이브러리인 Crawl4AI를 사용하여, 웹 페이지의 HTML 구조에서 특정 데이터를 추출하기 위한 JSON 스키마를 생성하고 적용하는 과정
- 복잡한 코딩 없이 AI에게 HTML 샘플을 보여주고, 앞으로 이 사이트에서 데이터를 어떻게 뽑아올지 규칙(Schema)을 만들게 시키는 코드

In [19]:
from google.colab import userdata
openai_token = userdata.get('openapi')

In [20]:
from crawl4ai.extraction_strategy import JsonCssExtractionStrategy
from crawl4ai import LLMConfig

# Generate a schema (one-time cost)
html = "<div class='product'><h2>Gaming Laptop</h2><span class='price'>$999.99</span></div>"


# Using OpenAI (requires API token)
schema = JsonCssExtractionStrategy.generate_schema(
    html,
    llm_config = LLMConfig(provider="openai/gpt-4o",api_token= openai_token)  # Required for OpenAI
)

# Use the schema for fast, repeated extractions
strategy = JsonCssExtractionStrategy(schema)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Rate limit error: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Waiting for 2 seconds before retrying...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Rate limit error: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Waiting for 4 seconds before retrying...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
Lit

Exception: Failed to generate schema: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.

In [ ]:
strategy.schema

{'name': 'Product Listing',
 'baseSelector': '.product',
 'fields': [{'name': 'product_name', 'selector': 'h2', 'type': 'text'},
  {'name': 'price', 'selector': '.price', 'type': 'text'}]}

In [21]:
import requests
from bs4 import BeautifulSoup

url = "https://comic.naver.com/webtoon/weekday"
res = requests.get(url)

soup = BeautifulSoup(res.text, "lxml")  # HTML 파싱

In [ ]:
# Using OpenAI (requires API token)
schema = JsonCssExtractionStrategy.generate_schema(
    soup,
    llm_config = LLMConfig(provider="openai/gpt-4o",api_token= openai_token)  # Required for OpenAI
)

# Use the schema for fast, repeated extractions
strategy = JsonCssExtractionStrategy(schema)

In [ ]:
strategy.schema

{'name': 'Naver Webtoon Page',
 'baseSelector': 'head',
 'fields': [{'name': 'title', 'selector': 'title', 'type': 'text'},
  {'name': 'favicon',
   'selector': "link[rel='icon']",
   'type': 'attribute',
   'attribute': 'href'},
  {'name': 'canonical_url',
   'selector': "link[rel='canonical']",
   'type': 'attribute',
   'attribute': 'href'},
  {'name': 'google_site_verification',
   'selector': "meta[name='google-site-verification']",
   'type': 'attribute',
   'attribute': 'content'},
  {'name': 'charset',
   'selector': 'meta[charset]',
   'type': 'attribute',
   'attribute': 'charset'},
  {'name': 'og_type',
   'selector': "meta[property='og:type']",
   'type': 'attribute',
   'attribute': 'content'},
  {'name': 'og_author',
   'selector': "meta[property='og:article:author']",
   'type': 'attribute',
   'attribute': 'content'},
  {'name': 'og_author_url',
   'selector': "meta[property='og:article:author:url']",
   'type': 'attribute',
   'attribute': 'content'},
  {'name': 'og_ti

In [ ]:
# prompt: strategy 에서 text를 추출하려면

# ... (Your existing code)

# Assuming 'strategy' is your JsonCssExtractionStrategy instance
if strategy and strategy.schema:
  print("Extracted text from schema:")
  for key, value in strategy.schema.items():
    print(f"Key: {key}, Value: {value['css_selector']}")
    # You might need to adapt this based on how your schema is structured

    # Example: Extract the text content of elements matching the CSS selector
    elements = soup.select(value['css_selector'])
    for element in elements:
      print(element.get_text(strip=True))



Extracted text from schema:


TypeError: string indices must be integers, not 'str'

In [ ]:
# Assuming 'strategy' is your JsonCssExtractionStrategy instance
if strategy and strategy.schema:
  print("Extracted data from schema:")
  for key, value in strategy.schema.items():
    print(f"Key: {key}, Value: {value}")  # Print the extracted value directly
    # If you need to access specific parts of the extracted data,
    # you'll need to inspect its structure and adjust the code accordingly.
    # For example, if 'value' is a dictionary, you might access specific fields like this:
    # if isinstance(value, dict) and 'text' in value:
    #   print(f"  Text: {value['text']}")

Extracted data from schema:
Key: name, Value: Naver Webtoon Page
Key: baseSelector, Value: head
Key: fields, Value: [{'name': 'title', 'selector': 'title', 'type': 'text'}, {'name': 'favicon', 'selector': "link[rel='icon']", 'type': 'attribute', 'attribute': 'href'}, {'name': 'canonical_url', 'selector': "link[rel='canonical']", 'type': 'attribute', 'attribute': 'href'}, {'name': 'google_site_verification', 'selector': "meta[name='google-site-verification']", 'type': 'attribute', 'attribute': 'content'}, {'name': 'charset', 'selector': 'meta[charset]', 'type': 'attribute', 'attribute': 'charset'}, {'name': 'og_type', 'selector': "meta[property='og:type']", 'type': 'attribute', 'attribute': 'content'}, {'name': 'og_author', 'selector': "meta[property='og:article:author']", 'type': 'attribute', 'attribute': 'content'}, {'name': 'og_author_url', 'selector': "meta[property='og:article:author:url']", 'type': 'attribute', 'attribute': 'content'}, {'name': 'og_title', 'selector': "meta[proper

In [ ]:
strategy.schema['name']

'Naver Webtoon Page'

In [ ]:
strategy.schema['baseSelector']

'head'